In [1]:
# Khai báo thư viện sử dụng
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re

In [2]:
# Hàm này truy cập một trang tìm kiếm và lấy URL của các tin đăng hợp lệ
def get_urls_from_page(page_url):
    real_listing_links_hrefs = [] 
    driver = None
    print(f"Đang quét trang: {page_url}")
    
    try:
        # Cấu hình Selenium
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')

        driver = webdriver.Chrome(options=chrome_options)
        driver.get(page_url)
        
        # Đợi tối đa 15 giây cho các thẻ tin đăng xuất hiện
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.js__card"))
        )
        
        # Phân tích HTML của trang bằng BeautifulSoup
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Trích xuất và lọc URL
        # Tìm tất cả các thẻ 'card' chứa thông tin tin đăng (bao gồm cả tin thật và quảng cáo)
        all_cards = soup.select('div.js__card')
        
        for card in all_cards:
            # Kiểm tra sự hiện diện của thông tin giá và diện tích là dấu hiệu tốt
            has_price = card.select_one('.re__card-config-price') is not None
            has_area = card.select_one('.re__card-config-area') is not None
            
            # Một cách kiểm tra khác cho quảng cáo là thuộc tính js-tracking-id trên thẻ <a> con
            link_tag = card.select_one('a.js__product-link-for-product-id')
            is_sponsored_by_tracking_id = False
            if link_tag and link_tag.has_attr('js-tracking-id') and 'srp-promote-verified-click-card' in link_tag['js-tracking-id']:
                is_sponsored_by_tracking_id = True

            # Nếu có giá và diện tích và không phải là tin được tài trợ rõ ràng qua tracking-id thì lấy
            if has_price and has_area and not is_sponsored_by_tracking_id:
                if link_tag:
                    href = "https://batdongsan.com.vn" + link_tag.get('href', 'N/A')
                    real_listing_links_hrefs.append(href)
                
        # Loại bỏ các link trùng lặp trong cùng một trang
        unique_real_links = list(dict.fromkeys(real_listing_links_hrefs))
        return unique_real_links
            
    except Exception as e:
        print(f"Lỗi khi truy cập {page_url}: {type(e).__name__} - {e}") 
        return [] # Trả về danh sách rỗng nếu có lỗi
    finally:
        # Đảm bảo trình duyệt luôn được đóng sau khi hoàn tất
        if driver:
            driver.quit()


# Hàm này cào dữ liệu chi tiết từ một URL tin đăng
def scrape_single_page(url, max_retries=3):
    driver = None

    # Vòng lặp để thử lại nếu gặp lỗi mạng hoặc trang tải chậm
    for attempt in range(max_retries):
        try:
            # Cấu hình Selenium
            chrome_options = Options()
            chrome_options.add_argument("--headless")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
            chrome_options.add_argument("--log-level=3")
            chrome_options.add_experimental_option('excludeSwitches', ['enable-logging'])
            
            driver = webdriver.Chrome(options=chrome_options)
            driver.get(url)

            # Đợi tối đa 15 giây cho tiêu đề tin đăng xuất hiện
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CLASS_NAME, "re__pr-title")))
            
            page_source = driver.page_source
            soup = BeautifulSoup(page_source, 'html.parser')

            # Trích xuất thông tin cơ bản 
            tieu_de = soup.find('h1', class_='re__pr-title').get_text(strip=True) if soup.find('h1', class_='re__pr-title') else "N/A"
            
            # Lấy địa chỉ, ưu tiên dòng 1, nếu không có thì lấy cha nối bằng dấu cách để không bị dính chữ
            dia_chi_line_1 = soup.select_one('.re__address-line-1')
            if dia_chi_line_1:
                dia_chi = dia_chi_line_1.get_text(strip=True)
            else:
                dia_chi_tag = soup.select_one('.re__address, .js__pr-address, .re__pr-short-description')
                dia_chi = dia_chi_tag.get_text(separator=' ', strip=True) if dia_chi_tag else "N/A"

            # Trích xuất thông tin từ bảng thông số kỹ thuật
            # Lấy tất cả các cặp (tiêu đề, giá trị) trong bảng
            specs_dict = {
                item.find('span', class_='re__pr-specs-content-item-title').get_text(strip=True): 
                item.find('span', class_='re__pr-specs-content-item-value').get_text(strip=True)
                for item in soup.select('.re__pr-specs-content-v2 .re__pr-specs-content-item')
                if item.find('span', class_='re__pr-specs-content-item-title') and item.find('span', class_='re__pr-specs-content-item-value')
            }

            gia = specs_dict.get('Khoảng giá', 'N/A')
            dien_tich_dat = specs_dict.get('Diện tích', 'N/A')
            phong_ngu = specs_dict.get('Số phòng ngủ', 'N/A')
            phong_tam = specs_dict.get('Số phòng tắm, vệ sinh', 'N/A')
            so_tang = specs_dict.get('Số tầng', 'N/A')
            phap_ly = specs_dict.get('Pháp lý', 'N/A')
            
            # Xử lý dữ liệu bị thiếu từ phần mô tả 
            if so_tang == "N/A" or phap_ly == "N/A":
                desc_tag = soup.find('div', class_='re__detail-content')
                if desc_tag:
                    desc_text = desc_tag.get_text(separator='\n')
                    if so_tang == "N/A":
                        tong_tang = 0
                        # Tìm mẫu tường minh nhất "X tầng" hoặc "X tấm" (cách gọi khác của tầng)
                        tang_match = re.search(r'(\d+)\s*(?:tầng|tấm)', desc_text, re.IGNORECASE)
                        if tang_match:
                            tong_tang = int(tang_match.group(1))
                        else:
                            # Tính toán từ "X lầu"
                            lau_match = re.search(r'(\d+)\s*lầu', desc_text, re.IGNORECASE)
                            if lau_match:
                                # Số tầng = số lầu + 1 (cho tầng trệt)
                                tong_tang = int(lau_match.group(1)) + 1
                            # Xử lý các trường hợp chữ như "trệt lầu"
                            elif re.search(r'trệt\s*[, ]\s*lầu', desc_text, re.IGNORECASE):
                                tong_tang = 2 # 1 trệt + 1 lầu

                        # Nếu tìm được số tầng từ "lầu", kiểm tra thêm "sân thượng"
                        if tong_tang > 0 and not tang_match:
                            if re.search(r'sân thượng', desc_text, re.IGNORECASE):
                                tong_tang += 1
                        
                        # Chỉ cập nhật nếu tìm thấy số tầng hợp lệ (lớn hơn 0)
                        if tong_tang > 0:
                            so_tang = f"{tong_tang} tầng"
            
                    # Xử lý pháp lý bằng regex trong phần mô tả nếu chưa tìm thấy
                    if phap_ly == "N/A":
                        # Kiểm tra sự tồn tại của "sổ hồng" và "sổ đỏ"
                        has_so_hong = re.search(r'sổ hồng', desc_text, re.IGNORECASE)
                        has_so_do = re.search(r'sổ đỏ', desc_text, re.IGNORECASE)

                        if has_so_hong and has_so_do:
                            phap_ly = "Sổ đỏ/ Sổ hồng"
                        elif has_so_hong:
                            phap_ly = "Sổ hồng"
                        elif has_so_do:
                            phap_ly = "Sổ đỏ" 

            # Trích xuất ngày đăng                 
            ngay_dang = next((item.find('span', class_='value').get_text(strip=True) 
                               for item in soup.select('.re__pr-config .re__pr-short-info-item')
                               if item.find('span', class_='title') and item.find('span', class_='value') and item.find('span', class_='title').get_text(strip=True) == 'Ngày đăng'), "N/A")
            
            # Trả về kết quả dưới dạng dictionary
            return {'tieu_de': tieu_de, 'gia': gia, 'dia_chi': dia_chi, 'dien_tich_dat': dien_tich_dat, 'phong_ngu': phong_ngu, 'phong_tam': phong_tam, 'so_tang': so_tang, 'phap_ly': phap_ly, 'ngay_dang': ngay_dang}
        except Exception as e:
            # Xử lý lỗi và in thông báo thử lại
            if attempt < max_retries:
                print(f"Lỗi khi cào URL {url} (lần {attempt + 1}/{max_retries}): {type(e).__name__} - Đang thử lại...")
                time.sleep(2) # Đợi 2 giây trước khi thử lại
            else:
                print(f"Lỗi khi cào URL {url} (tất cả {max_retries + 1} lần thất bại). Bỏ qua URL này.")
        finally:
            if driver:
                driver.quit()
                driver = None # Đặt lại driver để đảm bảo tạo mới cho lần thử lại
    return None # Trả về None nếu tất cả các lần thử đều thất bại


In [3]:
# Danh sách các quận cần cào
districts = [
    'quan-1', 'quan-2', 'quan-3', 'quan-4', 'quan-5', 'quan-6', 'quan-7', 'quan-8', 'quan-9',
    'quan-10', 'quan-11', 'quan-12', 'binh-tan', 'binh-thanh', 
    'go-vap', 'phu-nhuan', 'tan-binh', 'tan-phu'
    ]

# Số trang cần cào cho mỗi quận
pages_per_district = 35  # 35 trang x ~20 BĐS/trang = ~700 BĐS/quận


print("GIAI ĐOẠN 1: Bắt đầu thu thập URLs từ tất cả các quận")

# Sử dụng list để lưu trữ tất cả các URL sẽ được cào
all_urls_to_scrape = []
# URL cơ sở cho việc tìm kiếm
base_url = "https://batdongsan.com.vn/ban-nha-rieng-"

# Lặp qua từng quận trong danh sách đã định nghĩa
for district in districts:
    print(f"\n--- Đang quét: {district.upper()} ---")
    # Lặp qua từng trang trong phạm vi đã định nghĩa cho mỗi quận
    for page_num in range(1, pages_per_district + 1):
        # Tạo URL tìm kiếm hoàn chỉnh cho trang cụ thể
        # Trang đầu tiên không có hậu tố '/pX'
        search_url = f"{base_url}{district}" if page_num == 1 else f"{base_url}{district}/p{page_num}"
        
        # Gọi hàm để lấy tất cả các URL hợp lệ từ trang hiện tại
        urls_on_page = get_urls_from_page(search_url)
        if urls_on_page:
            # Thêm các URL vừa tìm được vào danh sách tổng
            all_urls_to_scrape.extend(urls_on_page)  
            print(f"=> Trang {page_num}: Tìm thấy {len(urls_on_page)} URLs (thêm {len(urls_on_page)} URLs mới)")
            print(f"=> Tổng cộng: {len(all_urls_to_scrape)} URLs.")
        else:
            # Nếu một trang không trả về URL nào, có thể đã hết trang
            print(f"=> Trang {page_num}: Không tìm thấy link nào, chuyển sang quận tiếp theo.")
            break 
        
        time.sleep(1) # Tạm dừng 1 giây để tránh bị chặn


all_urls_to_scrape = list(set(all_urls_to_scrape))  # Lọc trùng 

# Lưu danh sách URL vào file csv để dễ quản lý
pd.Series(all_urls_to_scrape, name='url').to_csv('../data/raw/all_urls.csv', index=False)


GIAI ĐOẠN 1: Bắt đầu thu thập URLs từ tất cả các quận

--- Đang quét: QUAN-1 ---
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1
=> Trang 1: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 20 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p2
=> Trang 2: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 40 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p3
=> Trang 3: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 60 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p4
=> Trang 4: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 80 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p5
=> Trang 5: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 100 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p6
=> Trang 6: Tìm thấy 20 URLs (thêm 20 URLs mới)
=> Tổng cộng: 120 URLs.
Đang quét trang: https://batdongsan.com.vn/ban-nha-rieng-quan-1/p7
=> Trang 7: Tìm thấy 20 U

In [ ]:
# Kiểm tra xem có URL nào được tìm thấy hay không
if not all_urls_to_scrape:
    print("\nKhông tìm thấy URL nào để cào. Chương trình kết thúc.")
else:
    print(f"\nĐã tìm thấy tổng cộng {len(all_urls_to_scrape)} URLs. ")
    print(f"\nGIAI ĐOẠN 2: Bắt đầu cào dữ liệu bất động sản từ những URLs đó")

    # Khởi tạo danh sách để lưu trữ tất cả dữ liệu cào được
    all_data = []

    # Tên file CSV để lưu kết quả
    output_filename = '../data/raw/batdongsan_raw.csv'

    # Ghi lại thời gian bắt đầu cào dữ liệu
    start_time = time.time()

    # Sử dụng ThreadPoolExecutor để thực hiện cào dữ liệu đa luồng, tăng tốc độ
    with ThreadPoolExecutor(max_workers=12) as executor:
        # Gửi tất cả các công việc (cào một URL) vào pool để thực thi.
        future_to_url = {executor.submit(scrape_single_page, url, max_retries=3): url for url in all_urls_to_scrape}
        
        # `as_completed` lặp qua các "công việc" ngay khi chúng hoàn thành, không theo thứ tự ban đầu
        for i, future in enumerate(as_completed(future_to_url)):
            # Lấy kết quả trả về từ hàm `scrape_single_page`
            result = future.result()

            # Nếu kết quả không phải là None (tức là cào thành công)
            if result:
                # Thêm dữ liệu vào danh sách tổng
                all_data.append(result)

                # Lưu dữ liệu vào file CSV ngay lập tức
                # Giúp bảo toàn dữ liệu đã cào được nếu chương trình bị lỗi giữa chừng
                df_single = pd.DataFrame([result])

                # Kiểm tra xem file CSV đã tồn tại hay chưa
                file_exists = os.path.isfile(output_filename)
                
                if not file_exists:
                    # Nếu file chưa tồn tại, tạo file mới và ghi cả header (tên cột).
                    df_single.to_csv(output_filename, mode='w', index=False, encoding='utf-8-sig')
                else:
                    # Nếu file đã tồn tại, ghi nối (append) dữ liệu vào cuối file mà không cần ghi header.
                    df_single.to_csv(output_filename, mode='a', header=False, index=False, encoding='utf-8-sig')

            # In ra tiến trình sau mỗi 10 URL
            if (i + 1) % 10 == 0:
                print(f"Hoàn thành {i + 1}/{len(all_urls_to_scrape)} URLs.")

    # Ghi lại thời gian kết thúc và tính toán tổng thời gian cào.
    end_time = time.time()
    print(f"\nTổng thời gian cào {len(all_data)} trang: {end_time - start_time:.2f} giây")

    # Hiển thị kết quả cuối cùng
    # Nếu có dữ liệu được cào thành công
    if all_data:
        # Tạo một DataFrame từ tất cả dữ liệu đã thu thập
        df = pd.DataFrame(all_data)
        print(f"\nDữ liệu đã được cào thành công và lưu vào file '{output_filename}'")
        print("\nXem trước 5 dòng dữ liệu đầu tiên:")
        print(df.head().to_string())
    else:
        print("\nKhông có dữ liệu nào được cào thành công.")


Đã tìm thấy tổng cộng 10114 URLs. 

GIAI ĐOẠN 2: Bắt đầu cào dữ liệu bất động sản từ những URLs đó
Hoàn thành 10/10114 URLs.
Hoàn thành 20/10114 URLs.
Hoàn thành 30/10114 URLs.
Hoàn thành 40/10114 URLs.
Hoàn thành 50/10114 URLs.
Hoàn thành 60/10114 URLs.
Hoàn thành 70/10114 URLs.
Hoàn thành 80/10114 URLs.
Hoàn thành 90/10114 URLs.
Hoàn thành 100/10114 URLs.
Hoàn thành 110/10114 URLs.
Hoàn thành 120/10114 URLs.
Hoàn thành 130/10114 URLs.
Hoàn thành 140/10114 URLs.
Hoàn thành 150/10114 URLs.
Hoàn thành 160/10114 URLs.
Hoàn thành 170/10114 URLs.
Hoàn thành 180/10114 URLs.
Hoàn thành 190/10114 URLs.
Hoàn thành 200/10114 URLs.
Hoàn thành 210/10114 URLs.
Hoàn thành 220/10114 URLs.
Hoàn thành 230/10114 URLs.
Hoàn thành 240/10114 URLs.
Hoàn thành 250/10114 URLs.
Hoàn thành 260/10114 URLs.
Hoàn thành 270/10114 URLs.
Hoàn thành 280/10114 URLs.
Hoàn thành 290/10114 URLs.
Hoàn thành 300/10114 URLs.
Hoàn thành 310/10114 URLs.
Hoàn thành 320/10114 URLs.
Hoàn thành 330/10114 URLs.
Hoàn thành 340/101

Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


Hoàn thành 2020/10114 URLs.
Hoàn thành 2030/10114 URLs.
Hoàn thành 2040/10114 URLs.
Hoàn thành 2050/10114 URLs.
Hoàn thành 2060/10114 URLs.
Hoàn thành 2070/10114 URLs.
Hoàn thành 2080/10114 URLs.
Hoàn thành 2090/10114 URLs.
Hoàn thành 2100/10114 URLs.
Hoàn thành 2110/10114 URLs.
Hoàn thành 2120/10114 URLs.
Hoàn thành 2130/10114 URLs.
Hoàn thành 2140/10114 URLs.
Hoàn thành 2150/10114 URLs.
Hoàn thành 2160/10114 URLs.
Hoàn thành 2170/10114 URLs.
Hoàn thành 2180/10114 URLs.
Hoàn thành 2190/10114 URLs.
Hoàn thành 2200/10114 URLs.
Hoàn thành 2210/10114 URLs.
Hoàn thành 2220/10114 URLs.
Lỗi khi cào URL https://batdongsan.com.vn/ban-nha-rieng-duong-so-19-phuong-8-12-67/chinh-chu-giam-ngay-450tr-ban-gap-1-tret-3-lau-st-goc-2mt-hem-19-p8-go-vap-pr45060994 (lần 1/3): TimeoutException - Đang thử lại...
Hoàn thành 2230/10114 URLs.
Hoàn thành 2240/10114 URLs.
Hoàn thành 2250/10114 URLs.
Hoàn thành 2260/10114 URLs.
Hoàn thành 2270/10114 URLs.
Hoàn thành 2280/10114 URLs.
Hoàn thành 2290/10114 URLs.
H

In [ ]:
import pandas as pd
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Đọc lại danh sách 10.114 URLs đã thu thập từ Giai đoạn 1
df_all_urls = pd.read_csv('../data/raw/all_urls.csv')
all_urls_to_scrape = df_all_urls['url'].tolist()

# 2. Đọc lượng dữ liệu đã cào để tính xem mình đã cào đến đâu
output_filename = '../data/raw/batdongsan_raw.csv'
if os.path.exists(output_filename):
    df_raw = pd.read_csv(output_filename)
    scraped_count = len(df_raw)
else:
    scraped_count = 0

print(f"Tổng số URL ban đầu: {len(all_urls_to_scrape)}")
print(f"Số lượng dữ liệu ĐÃ CÀO (Trong file CSV): {scraped_count}")

# 3. Lấy danh sách các URL còn lại để cào tiếp.
# (Cắt list để bỏ qua scraped_count định dạng đầu tiên)
remaining_urls = all_urls_to_scrape[scraped_count:]
print(f"Số lượng URL CẦN CÀO TIẾP GIAI ĐOẠN NÀY: {len(remaining_urls)}\n")
print("BẮT ĐẦU CÀO TIẾP VÀ NỐI VÀO DATA CŨ...")

# 4. Bắt đầu lại tiến trình cào đa luồng cho các URL còn lại
if len(remaining_urls) > 0:
    all_data = [] # Nếu muốn lưu lại batch này trong memory
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=12) as executor:
        # Đẩy các URL còn lại vào hàm
        future_to_url = {executor.submit(scrape_single_page, url, max_retries=3): url for url in remaining_urls}
        
        # Biến đếm cho lần cào bù này
        for i, future in enumerate(as_completed(future_to_url)):
            result = future.result()
            
            if result:
                all_data.append(result)
                df_single = pd.DataFrame([result])
                
                # CỰC KỲ QUAN TRỌNG: Ghi nối append (mode='a') vào file cũ, không ghi đè và KHÔNG ghi lại phần header
                df_single.to_csv(output_filename, mode='a', header=False, index=False, encoding='utf-8-sig')

            # In ra tiến trình (cộng dồn số luồng đã có từ đợt trước)
            current_total = scraped_count + i + 1
            if current_total % 10 == 0:
                print(f"Hoàn thành {current_total}/{len(all_urls_to_scrape)} URLs.")

    end_time = time.time()
    print(f"\nTổng thời gian cào {len(remaining_urls)} trang còn lại: {end_time - start_time:.2f} giây")
else:
    print("\nBạn đã cào xong toàn bộ danh sách URL!")


Tổng số URL ban đầu: 10114
Số lượng dữ liệu ĐÃ CÀO (Trong file CSV): 6321
Số lượng URL CẦN CÀO TIẾP GIAI ĐOẠN NÀY: 3793

BẮT ĐẦU CÀO TIẾP VÀ NỐI VÀO DATA CŨ...
Hoàn thành 6330/10114 URLs.
Hoàn thành 6340/10114 URLs.
Hoàn thành 6350/10114 URLs.
Hoàn thành 6360/10114 URLs.
Hoàn thành 6370/10114 URLs.
Lỗi khi cào URL https://batdongsan.com.vn/ban-nha-rieng-duong-hoa-hung-phuong-15-3-62/quan-10-gan-vong-xoay-dan-chu-kts-thiet-ke-57m2-9-5-x-6-3-tang-hem-1-sat-mat-tien-pr45241625 (lần 1/3): SessionNotCreatedException - Đang thử lại...
Hoàn thành 6380/10114 URLs.
Hoàn thành 6390/10114 URLs.
Hoàn thành 6400/10114 URLs.
Hoàn thành 6410/10114 URLs.
Hoàn thành 6420/10114 URLs.
Hoàn thành 6430/10114 URLs.
Hoàn thành 6440/10114 URLs.
Hoàn thành 6450/10114 URLs.
Hoàn thành 6460/10114 URLs.
Hoàn thành 6470/10114 URLs.
Hoàn thành 6480/10114 URLs.
Hoàn thành 6490/10114 URLs.
Hoàn thành 6500/10114 URLs.
Hoàn thành 6510/10114 URLs.
Hoàn thành 6520/10114 URLs.
Hoàn thành 6530/10114 URLs.
Hoàn thành 6540/1

In [3]:
import pandas as pd
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Đọc lại danh sách 10.114 URLs đã thu thập từ Giai đoạn 1
df_all_urls = pd.read_csv('../data/raw/all_urls.csv')
all_urls_to_scrape = df_all_urls['url'].tolist()

# 2. Đọc lượng dữ liệu đã cào để tính xem mình đã cào đến đâu
output_filename = '../data/raw/batdongsan_raw.csv'
if os.path.exists(output_filename):
    df_raw = pd.read_csv(output_filename)
    scraped_count = len(df_raw)
else:
    scraped_count = 0

print(f"Tổng số URL ban đầu: {len(all_urls_to_scrape)}")
print(f"Số lượng dữ liệu ĐÃ CÀO (Trong file CSV): {scraped_count}")

# 3. Lấy danh sách các URL còn lại để cào tiếp.
# (Cắt list để bỏ qua scraped_count định dạng đầu tiên)
remaining_urls = all_urls_to_scrape[scraped_count:]
print(f"Số lượng URL CẦN CÀO TIẾP GIAI ĐOẠN NÀY: {len(remaining_urls)}\n")
print("BẮT ĐẦU CÀO TIẾP VÀ NỐI VÀO DATA CŨ...")

# 4. Bắt đầu lại tiến trình cào đa luồng cho các URL còn lại
if len(remaining_urls) > 0:
    all_data = [] # Nếu muốn lưu lại batch này trong memory
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=12) as executor:
        # Đẩy các URL còn lại vào hàm
        future_to_url = {executor.submit(scrape_single_page, url, max_retries=3): url for url in remaining_urls}
        
        # Biến đếm cho lần cào bù này
        for i, future in enumerate(as_completed(future_to_url)):
            result = future.result()
            
            if result:
                all_data.append(result)
                df_single = pd.DataFrame([result])
                
                # CỰC KỲ QUAN TRỌNG: Ghi nối append (mode='a') vào file cũ, không ghi đè và KHÔNG ghi lại phần header
                df_single.to_csv(output_filename, mode='a', header=False, index=False, encoding='utf-8-sig')

            # In ra tiến trình (cộng dồn số luồng đã có từ đợt trước)
            current_total = scraped_count + i + 1
            if current_total % 10 == 0:
                print(f"Hoàn thành {current_total}/{len(all_urls_to_scrape)} URLs.")

    end_time = time.time()
    print(f"\nTổng thời gian cào {len(remaining_urls)} trang còn lại: {end_time - start_time:.2f} giây")
else:
    print("\nBạn đã cào xong toàn bộ danh sách URL!")


Tổng số URL ban đầu: 10114
Số lượng dữ liệu ĐÃ CÀO (Trong file CSV): 7729
Số lượng URL CẦN CÀO TIẾP GIAI ĐOẠN NÀY: 2385

BẮT ĐẦU CÀO TIẾP VÀ NỐI VÀO DATA CŨ...
Hoàn thành 7730/10114 URLs.
Hoàn thành 7740/10114 URLs.
Hoàn thành 7750/10114 URLs.
Hoàn thành 7760/10114 URLs.
Hoàn thành 7770/10114 URLs.
Hoàn thành 7780/10114 URLs.
Hoàn thành 7790/10114 URLs.
Hoàn thành 7800/10114 URLs.
Hoàn thành 7810/10114 URLs.
Hoàn thành 7820/10114 URLs.
Hoàn thành 7830/10114 URLs.
Hoàn thành 7840/10114 URLs.
Hoàn thành 7850/10114 URLs.
Hoàn thành 7860/10114 URLs.
Hoàn thành 7870/10114 URLs.
Hoàn thành 7880/10114 URLs.
Hoàn thành 7890/10114 URLs.
Hoàn thành 7900/10114 URLs.
Hoàn thành 7910/10114 URLs.
Hoàn thành 7920/10114 URLs.
Lỗi khi cào URL https://batdongsan.com.vn/ban-nha-rieng-duong-hiep-thanh-45-phuong-tan-thoi-hiep-64/ben-hong-truong-vo-truong-toan-ngang-5-6m-cuc-hiem-4-8-ty-pr45423778 (lần 1/3): SessionNotCreatedException - Đang thử lại...
Hoàn thành 7930/10114 URLs.
Hoàn thành 7940/10114 URLs.